# 0. import libraries

In [1]:
# @title Colab Setup Environment

try:
    import google.colab

    # Remove existing directory to ensure clean clone on re-run
    !rm -rf repository/circuit-tracer

    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login

    sys.path.append("repository/circuit-tracer")
    sys.path.append("repository/circuit-tracer/demos")
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
# ── 1. HuggingFace model for generation ──────────────────────────────────────
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")

# ── 2. circuit_tracer model for attribution (replaces HF model entirely) ─────
model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", dtype=torch.bfloat16, backend="transformerlens"
)

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


# 1. getting attributes from each token step
based on attribute_demo  
**can skip to step 2, features are saved in** `demos\graph_files`

In [3]:
max_n_logits = 10  # How many logits to attribute from, max. We attribute to min(max_n_logits, n_logits_to_reach_desired_log_prob); see below for the latter
desired_logit_prob = 0.95  # Attribution will attribute from the minimum number of logits needed to reach this probability mass (or max_n_logits, whichever is lower)
max_feature_nodes = 1024  # Only attribute from this number of feature nodes, max. Lower is faster, but you will lose more of the graph. None means no limit.
batch_size = 32  # Batch size when attributing
offload = "cpu"
verbose = True  # Whether to display a tqdm progress bar and timing report

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
# ── 3. Generation loop (same as before, but using ReplacementModel) ───────────
base_prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"
input_ids = tokenizer(base_prompt, return_tensors="pt")["input_ids"]
STOP_IDS = {108, 235265, tokenizer.eos_token_id}
token_trace = []

for step in range(20):
    with torch.no_grad():
      out = model(input_ids)

    logits_last = out[0, -1, :].float()  # out is already the logits tensor
    probs = F.softmax(logits_last, dim=-1)
    top_probs, top_ids = torch.topk(probs, 10)
    next_id = top_ids[0].item()

    token_trace.append({
        "step": step,
        "token_id": next_id,
        "token_str": tokenizer.decode([next_id]),
        "chosen_prob": top_probs[0].item(),
        "chosen_logit": logits_last[next_id].item(),
        "top10": [{"token": tokenizer.decode([tid.item()]), "prob": p.item()}
                  for tid, p in zip(top_ids, top_probs)],
    })

    if next_id in STOP_IDS:
        break

    input_ids = torch.cat(
        [input_ids, torch.tensor([[next_id]])], dim=1
    )

print("Generated:", "".join(t["token_str"] for t in token_trace if t["token_id"] not in STOP_IDS))

# ── 4. Attribution loop — token_trace flows straight in ──────────────────────
for i, token in enumerate(token_trace):
    if token["token_id"] in STOP_IDS:
        break

    tokens_so_far = "".join(t["token_str"] for t in token_trace[:i])
    prompt_at_step = base_prompt + tokens_so_far

    graph = attribute(
        prompt=prompt_at_step,
        model=model,
        max_n_logits=max_n_logits,
        desired_logit_prob=desired_logit_prob,
        max_feature_nodes=max_feature_nodes,   # ← missing
        batch_size=batch_size,
        offload=offload,                        # ← use the variable, not hardcoded "cpu"
        verbose=verbose,
    )

    slug = f"step-{i:02d}-{token['token_str'].strip().replace(' ', '_')}"
    create_graph_files(
        graph_or_path=graph,
        slug=slug,
        output_path="./graph_files/gemma-2-2b-1/",
        node_threshold=0.8,
        edge_threshold=0.98,
    )
    print(f"✓ step {i:02d} → '{token['token_str']}'  saved as '{slug}'")

Phase 0: Precomputing activations and vectors


Generated: He saw a carrot and had to grab it


Precomputation completed in 0.79s
Found 15408 active features
Phase 1: Running forward pass
Forward pass completed in 2.44s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.6094
Will include 1024 of 15408 feature nodes
Input vectors built in 0.52s
Phase 3: Computing logit attributions
c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\circuit_tracer\attribution\context_transformerlens.py:223: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  self._resid_activations[last_layer].backward(
Logit attributions completed in 0.77s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [00:17<00:00, 60.01it/s]
Feature attributions completed in 17.07s
Attribution completed in 21.60s
Pha

✓ step 00 → 'He'  saved as 'step-00-He'


Precomputation completed in 0.59s
Found 16293 active features
Phase 1: Running forward pass
Forward pass completed in 2.41s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.5117
Will include 1024 of 16293 feature nodes
Input vectors built in 0.37s
Phase 3: Computing logit attributions
Logit attributions completed in 0.69s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [00:18<00:00, 55.10it/s]
Feature attributions completed in 18.59s
Attribution completed in 22.67s
Phase 0: Precomputing activations and vectors


...

✓ step 06 → ' to'  saved as 'step-06-to'


Precomputation completed in 0.92s
Found 22262 active features
Phase 1: Running forward pass
Forward pass completed in 3.52s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.8398
Will include 1024 of 22262 feature nodes
Input vectors built in 0.53s
Phase 3: Computing logit attributions
Logit attributions completed in 0.90s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [00:26<00:00, 39.26it/s]
Feature attributions completed in 26.09s
Attribution completed in 31.97s
Phase 0: Precomputing activations and vectors


✓ step 07 → ' grab'  saved as 'step-07-grab'


Precomputation completed in 0.66s
Found 23785 active features
Phase 1: Running forward pass
Forward pass completed in 3.61s
Phase 2: Building input vectors
Using 1 salient logits with cumulative probability 0.9883
Will include 1024 of 23785 feature nodes
Input vectors built in 0.33s
Phase 3: Computing logit attributions
Logit attributions completed in 1.00s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [00:27<00:00, 36.84it/s]
Feature attributions completed in 27.79s
Attribution completed in 33.41s


✓ step 08 → ' it'  saved as 'step-08-it'


In [10]:
import json
from pathlib import Path

total = sum(
    sum(1 for node in json.loads(p.read_text())["nodes"]
        if not node["is_target_logit"] and node["influence"] is not None)
    for p in sorted(Path("./graph_files/gemma-2-2b-2").glob("step-*.json"))
)
print(f"{total} nodes → ~{total * 0.1 / 60:.1f} minutes")

8147 nodes → ~13.6 minutes


In [12]:
# extracting clerp descriptions and adding to graph files

import json, time, requests
from pathlib import Path

def fetch_clerp(layer, feature):
    url = f"https://www.neuronpedia.org/api/feature/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feature}"
    r = requests.get(url)
    if r.ok:
        data = r.json()
        descs = data.get("explanations", [])
        if descs:
            return descs[0].get("description", "")
    return ""

for json_path in sorted(Path("./graph_files/gemma-2-2b-2").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    for node in data['nodes']:
        if node['is_target_logit'] or node['influence'] is None:
            continue
        parts = node['node_id'].split('_')
        feat = parts[1]
        layer = node['layer']
        node['clerp'] = fetch_clerp(layer, feat)
        time.sleep(0.1)  # be nice to the API
    json_path.write_text(json.dumps(data))
    print(f"done: {json_path.name}")

done: step-00-And.json
done: step-01-saw.json
done: step-02-the.json
done: step-03-world.json
done: step-04-in.json
done: step-05-a.json
done: step-06-new.json
done: step-07-way.json


# 2. attribute graph visualization

In [5]:
import json
from pathlib import Path

graph_dir = Path("./graph_files/gemma-2-2b-1")

for json_path in sorted(graph_dir.glob("step-*.json")):
    data = json.loads(json_path.read_text())
    
    # collect all unique layers used by transcoder nodes
    layers = sorted(set(
        int(node["layer"])
        for node in data["nodes"]
        if not node["is_target_logit"] and node.get("feature_type") == "cross layer transcoder"
    ))
    
    transcoder_list = [f"gemma-2-2b/{l}-gemmascope-transcoder-16k" for l in layers]
    data["metadata"]["transcoder_list"] = transcoder_list
    
    json_path.write_text(json.dumps(data))
    print(f"{json_path.name}: {len(layers)} layers → {transcoder_list[:3]}...")

# also patch the manifest
manifest_path = graph_dir / "graph-metadata.json"
manifest = json.loads(manifest_path.read_text())
for entry in manifest["graphs"]:
    slug = entry["slug"]
    step_path = graph_dir / f"{slug}.json"
    if step_path.exists():
        step_data = json.loads(step_path.read_text())
        entry["transcoder_list"] = step_data["metadata"]["transcoder_list"]

manifest_path.write_text(json.dumps(manifest, indent=2))
print("manifest patched")

step-00-He.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-01-saw.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-02-a.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-03-carrot.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-04-and.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-05-had.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-06-to.json: 26 layers → ['gemma-2-2

In [6]:
import json
from pathlib import Path

data = {"graphs": []}  # ← Add this line to initialize

for json_path in sorted(Path("./graph_files/gemma-2-2b-1").glob("step-*.json")):
    graph_data = json.loads(json_path.read_text())
    data["graphs"].append(graph_data["metadata"])

Path("./graph_files/gemma-2-2b-1/graph-metadata.json").write_text(json.dumps(data, indent=2))
print("done")

done


In [7]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame

port = 8046
server = serve(data_dir="./graph_files/gemma-2-2b-1/", port=port)

print(f"http://localhost:{port}/index.html")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))

http://localhost:8046/index.html


## 2.1 feature analysis based on circuit

In [8]:
import json
from pathlib import Path
from collections import defaultdict

# collect top 10 nodes per step
top_features = set()

for json_path in sorted(Path("./graph_files/gemma-2-2b-1/").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    nodes = [n for n in data['nodes'] if not n['is_target_logit'] and n['influence'] is not None]
    nodes = sorted(nodes, key=lambda x: x['influence'], reverse=True)[:10]
    for node in nodes:
        parts = node['node_id'].split('_')
        top_features.add((node['layer'], parts[1]))

print(f"Unique top features across all steps: {len(top_features)}")

Unique top features across all steps: 81


In [9]:
data = json.loads(Path("./graph_files/qwen3-4b/step-00-But.json").read_text())
layers = set(n['layer'] for n in data['nodes'])
print(sorted(layers))

['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '37', '4', '5', '6', '7', '8', '9', 'E']


In [10]:
import requests, time

def fetch_clerp(layer, feature):
    url = f"https://www.neuronpedia.org/api/feature/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feature}"
    r = requests.get(url)
    if r.ok:
        descs = r.json().get("explanations", [])
        if descs:
            return descs[0].get("description", "")
    return ""

clerp_cache = {}
for i, (layer, feat) in enumerate(top_features):
    key = f"L{layer}F{feat}"
    clerp_cache[key] = fetch_clerp(layer, feat)
    time.sleep(0.1)
    if i % 10 == 0:
        print(f"{i}/{len(top_features)} fetched...")

print("done")

0/81 fetched...
10/81 fetched...
20/81 fetched...
30/81 fetched...
40/81 fetched...
50/81 fetched...
60/81 fetched...
70/81 fetched...
80/81 fetched...
done


In [11]:
for key, clerp in sorted(clerp_cache.items()):
    if clerp:  # only show features that have descriptions
        print(f"{key}: {clerp}")

L0F10219:  the indefinite article "a"
L0F10918:  the word "seen" within the context of research and scientific papers
L0F11612:  technical or scientific terms, especially in a research context
L0F12077:  the word "node" in what seems to be a description of a computer network
L0F12191: present tense verbs
L0F12640:  the letter 'A' by itself, possibly indicating a header or section marker
L0F12924:  the word "and", with a slightly stronger activation for lists of items
L0F15768:  terms related to chemistry, pharmaceuticals, and material science
L0F1895:  the word "slow" along with the word "little"
L0F2413:  instances of something being tried, whether physically, legally, or experimentally
L0F2901:  the word "possession" and sometimes the numbers five and ten
L0F4133:  variations of the word "secret."
L0F4470:  references to religion, religious figures, or important religious places
L0F4876:  the word "handy" or verbs with a positive connotation
L0F6134:  words, especially adjectives, re

In [12]:
# find which steps each poetry feature appears in
poetry_features = {k for k, v in clerp_cache.items() if v and any(
    word in v.lower() for word in ['poem', 'poet', 'lyric', 'rhym', 'song', 'verse']
)}

print("Poetry-related features:", poetry_features)
print()

for json_path in sorted(Path("./graph_files/qwen3-4b/").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    slug = data['metadata']['slug']
    
    for node in data['nodes']:
        if node['is_target_logit'] or node['influence'] is None:
            continue
        parts = node['node_id'].split('_')
        key = f"L{node['layer']}F{parts[1]}"
        if key in poetry_features:
            print(f"  {slug}  {key}  influence={node['influence']:.4f}  {clerp_cache[key]}")

Poetry-related features: {'L5F6651', 'L15F13008', 'L4F4973', 'L25F2242', 'L3F1614', 'L7F3788', 'L4F1360', 'L7F11723', 'L4F4619'}



# 3. intervention
based on intervention_demo

In [13]:
from collections import namedtuple
from functools import partial

# display functions
from circuit_tracer.utils.demo_utils import display_topk_token_predictions, display_generations_comparison

In [14]:
Feature = namedtuple('Feature', ['layer', 'pos', 'feature_idx'])

# a display function that needs the model's tokenizer
display_topk_token_predictions = partial(display_topk_token_predictions, tokenizer=model.tokenizer)

In [15]:
supernode_features = [
    Feature(layer=5,pos=-1,feature_idx=6651),
    Feature(layer=15,pos=-1,feature_idx=13008),
    Feature(layer=4,pos=-1,feature_idx=4973),
    Feature(layer=25,pos=-1,feature_idx=2242),
    Feature(layer=3,pos=-1,feature_idx=1614),
    Feature(layer=7,pos=-1,feature_idx=3788),
    Feature(layer=4,pos=-1,feature_idx=1360),
    Feature(layer=7,pos=-1,feature_idx=11723),
    Feature(layer=4,pos=-1,feature_idx=4619)
]

intervention_tuples = [(*supernode_feature, 0.0) for supernode_feature in supernode_features]

In [16]:
s = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

In [17]:
with torch.inference_mode():
    original_logits, _  = model.feature_intervention(s, [])
    new_logits, _ = model.feature_intervention(s, intervention_tuples)

display_topk_token_predictions(s, original_logits, new_logits)

Token,Probability,Distribution
He,0.152,15.2%
But,0.152,15.2%
And,0.072,7.2%
The,0.044,4.4%
Then,0.039,3.9%
Token,Probability,Distribution
He,0.160,16.0%
But,0.160,16.0%
And,0.076,7.6%
The,0.041,4.1%


In [18]:
from circuit_tracer.utils.demo_utils import display_generations_comparison

# suppress both candidate features at the newline position (-1)
intervention_tuples = [
    (5, -1, 6651, 0.0),
    (15, -1, 13008, 0.0),
    (4, -1, 4973, 0.0),
    (25, -1, 2242, 0.0),
    (3, -1, 1614, 0.0),
    (7, -1, 3788, 0.0),
    (4, -1, 1360, 0.0),
    (7, -1, 11723, 0.0),
    (4, -1, 4619, 0.0),
]

print("Generating with original features...")
pre = [model.feature_intervention_generate(s, [], do_sample=False, verbose=True)[0]]

print("\nGenerating with interventions...")
post = [model.feature_intervention_generate(s, intervention_tuples, do_sample=False, verbose=True)[0]]

display_generations_comparison(s, pre, post)

Generating with original features...


  0%|          | 0/10 [00:00<?, ?it/s]


Generating with interventions...


  0%|          | 0/10 [00:00<?, ?it/s]